# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-6-luna"   # 호출마다 reasoning_effort="none" 을 함께 넣는다 (교재 5.3)
print("client ready:", MODEL)

client ready: gpt-6-luna


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [21]:
resp = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "고양이를 한 문장으로 묘사해"},
    ],
    temperature=0.9,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

고양이는 호기심 많고 우아하며, 따뜻한 햇살과 조용한 낮잠을 사랑하는 작은 동반자입니다.
usage: CompletionUsage(completion_tokens=36, prompt_tokens=34, total_tokens=70, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [10]:
stream = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    messages=[{"role": "user", "content": "LLM 을 13줄로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

1. LLM은 ‘대규모 언어 모델(Large Language Model)’의 약자입니다.
2. 방대한 양의 글을 학습해 언어의 패턴을 익힙니다.
3. 입력된 문맥을 바탕으로 다음에 올 말을 예측합니다.
4. 이렇게 단어를 이어 가며 문장이나 답변을 만들어 냅니다.
5. 질문 답변, 요약, 번역, 글쓰기 등에 활용됩니다.
6. 코드 작성이나 정보 정리에도 도움을 줄 수 있습니다.
7. 사람처럼 대화하는 듯 보이지만, 사람처럼 생각하거나 이해하는 것은 아닙니다.
8. 학습 자료의 패턴을 바탕으로 그럴듯한 답을 생성합니다.
9. 그래서 사실과 다른 내용을 자신 있게 말할 수 있습니다.
10. 최신 정보나 전문적인 내용은 별도로 확인하는 것이 좋습니다.
11. 질문을 구체적으로 할수록 원하는 답을 얻기 쉽습니다.
12. 개인정보나 민감한 정보를 입력할 때는 주의해야 합니다.
13. 한마디로, LLM은 언어를 다루는 강력한 예측·생성 도구입니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [11]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    55
0    45
Name: count, dtype: int64


,clean_text,label
0,사진과 색이 완전히 달라서 실망했어요,0
1,사이즈도 딱 맞고 마감이 깔끔해요,1
2,색감이 사진이랑 똑같아서 만족합니다,1
3,사진과 색이 완전히 달라서 실망했어요,0
4,재구매 의사 100 강추합니다,1


In [ ]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        temperature=0,
        response_format={"type": "text"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    print(data)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
print(sample.loc[0, "clean_text"])
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

사진과 색이 완전히 달라서 실망했어요
{'label': 0}
라벨: 0 / 토큰: 83


In [17]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8588


### sklearn(실습 2) 과 비교 — 정확도

In [18]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 0.990
              precision    recall  f1-score   support

           0      1.000     0.978     0.989        45
           1      0.982     1.000     0.991        55

    accuracy                          0.990       100
   macro avg      0.991     0.989     0.990       100
weighted avg      0.990     0.990     0.990       100



## 3) Task 3 — 비용 측정 + 1만 건 추산

In [ ]:
# gpt-6-luna 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.10/M, 출력 0.50/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.10 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
# TODO: ML vs LLM — '언제 무엇을 쓸지' 한 줄 결론을 적는다

## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*